# Multimodality

Let's copy an image from a public bucket with examples first:

In [1]:
# setting the environment variables, the keys
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

from config import set_environment

# for the keys - as explained early in chapter 2
set_environment()

In [2]:
# !gsutil cp gs://cloud-samples-data/generative-ai/image/boats.jpeg .

We can send it as bytes and ask the model to describe it:

In [3]:
import base64
from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import HumanMessage

llm = ChatVertexAI(model="gemini-2.5-flash")

with open("boats.jpeg", 'rb') as image_file:
    image_bytes = image_file.read()
    base64_bytes = base64.b64encode(image_bytes).decode("utf-8")

prompt = [
    {"type": "text", "text": "Describe the image: "},
    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_bytes}"}}
]

response = llm.invoke([HumanMessage(content=prompt)])
print(response.content)

This image captures a vibrant scene on a wide river or bay, featuring two boats in the foreground and a prominent city skyline with a distinctive bridge in the background, all under an overcast sky.

In the immediate foreground, positioned to the right, is a detailed pontoon boat. It features light blue and white cushioned seating, a dark green canvas bimini top, and black side panels. The pontoon hulls themselves are silver with a subtle light green stripe along the waterline. An outboard motor is visible at the stern, and a steering wheel can be seen within the boat, suggesting a control console. A registration number, "DC 1056 BD," is clearly visible on its side.

Further back and to the left, a smaller white powerboat or fishing boat is moored. It also has an outboard motor and a light-colored interior. A round, cream-colored buoy floats nearby in the water. Scattered further out are additional smaller white buoys.

The water itself is dark, likely due to depth and the cloudy condi

We can also do the same with videos:

In [7]:
video_uri = "gs://cloud-samples-data/generative-ai/video/animals.mp4"
prompt = [
    {"type": "text", "text": "Describe the video in a few sentences."},
    {"type": "media", "file_uri": video_uri, "mime_type": "video/mp4"},
]

response = llm.invoke([HumanMessage(content=prompt)])
print(response.content)

Inspired by Disney's Zootopia, this video showcases a unique collaboration between Google Photos and the Los Angeles Zoo, where real animals get to use technology to take their own "selfies." Special animal-proof cameras, branded with Google Photos, were strategically placed in various enclosures, capturing fascinating close-up photos of giraffes, tigers, elephants, and otters as they playfully interacted with the devices. The captured images are automatically backed up to Google Photos and shared, providing the public with an unprecedented and personal look into the animals' lives.


Also we can define an offset (a piece of video to be processed by the model):

In [8]:
offset_hint = {
    "start_offset": {"seconds": 10},
    "end_offset": {"seconds": 20},
}

prompt = [
    {"type": "text", "text": "Describe the video in a few sentences."},
    {
        "type": "media",
        "file_uri": video_uri,
        "mime_type": "video/mp4",
        "video_metadata": offset_hint
    },
]

response = llm.invoke([HumanMessage(content=prompt)])

We can also use prompt substitution to pass bytes to the prompt:

In [5]:
image_uri = "gs://cloud-samples-data/generative-ai/image/boats.jpeg"

In [9]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "user",
            [
                {
                    "type": "image_url",
                    "image_url": {"url": "data:image/jpeg;base64,{image_bytes_str}"},
                }
            ],
        )
    ]
)
prompt.invoke({"image_bytes_str": "test-url"})

ChatPromptValue(messages=[HumanMessage(content=[{'type': 'image_url', 'image_url': {'url': 'data:image/jpeg;base64,test-url'}}], additional_kwargs={}, response_metadata={})])